In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch_numopt
import numpy as np
import matplotlib.pyplot as plt
from torch.utils.data import DataLoader, TensorDataset
from sklearn.datasets import *
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, mean_absolute_error
from train_loop import train_loop

In [2]:
device = "cuda" if torch.cuda.is_available() else "cpu"

In [3]:
class Net(nn.Module):
    def __init__(self, input_size, device="cpu"):
        super().__init__()
        self.f1 = nn.Linear(input_size, 10, device=device)
        self.f2 = nn.Linear(10, 20, device=device)
        self.f3 = nn.Linear(20, 20, device=device)
        self.f4 = nn.Linear(20, 10, device=device)
        self.f5 = nn.Linear(10, 1, device=device)

        self.activation = nn.ReLU()
        # self.activation = nn.Sigmoid()

    def forward(self, x):
        x = self.activation(self.f1(x))
        x = self.activation(self.f2(x))
        x = self.activation(self.f3(x))
        x = self.activation(self.f4(x))
        x = self.f5(x)

        return x

In [4]:
# X, y = load_diabetes(return_X_y=True, scaled=False)
# X, y = make_regression(n_samples=1000, n_features=100)
X, y = make_friedman1(n_samples=1000, noise=1e-2)

X_scaler = MinMaxScaler()
X = X_scaler.fit_transform(X)
X = torch.Tensor(X).to(device)

y_scaler = MinMaxScaler()
y = y_scaler.fit_transform(y.reshape((-1, 1)))
y = torch.Tensor(y).to(device)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)
print(X_train.shape, y_train.shape)
print(X_test.shape, y_test.shape)

torch_data = TensorDataset(X_train, y_train)
data_loader = DataLoader(torch_data, batch_size=1000)

torch.Size([800, 10]) torch.Size([800, 1])
torch.Size([200, 10]) torch.Size([200, 1])


In [5]:
model = Net(input_size=X.shape[1], device=device)
loss_fn = nn.MSELoss()

opt = torch_numopt.ConjugateGradientLS(
    model=model,
    cg_method="FR",
    max_iter=10000,
    tol=1e-100
)

model, loss_history = train_loop(
    model,
    loss_fn,
    opt,
    data_loader,
    epochs=1000,
    max_patience=50
)

epoch:  0, loss: 0.2345711886882782
epoch:  1, loss: 0.1288260817527771
epoch:  2, loss: 0.04051843658089638
epoch:  3, loss: 0.040222588926553726
epoch:  4, loss: 0.039943061769008636
epoch:  5, loss: 0.039824046194553375
epoch:  6, loss: 0.039823755621910095
epoch:  7, loss: 0.03969644382596016
epoch:  8, loss: 0.03969623148441315
epoch:  9, loss: 0.03951464965939522
epoch:  10, loss: 0.039513006806373596
epoch:  11, loss: 0.03920648247003555
epoch:  12, loss: 0.0391850620508194
epoch:  13, loss: 0.03850702941417694
epoch:  14, loss: 0.03850702941417694
epoch:  15, loss: 0.037879377603530884
epoch:  16, loss: 0.037879377603530884
epoch:  17, loss: 0.037165626883506775
epoch:  18, loss: 0.037165626883506775
epoch:  19, loss: 0.03625113144516945
epoch:  20, loss: 0.03625113144516945
epoch:  21, loss: 0.03505529835820198
epoch:  22, loss: 0.03505529463291168
epoch:  23, loss: 0.033254023641347885
epoch:  24, loss: 0.033254023641347885
epoch:  25, loss: 0.028823496773838997
epoch:  26, l

In [6]:
pred_train = model.forward(X_train).cpu().detach()
pred_test = model.forward(X_test).cpu().detach()
print(f"Train metrics: R2 = {r2_score(pred_train, y_train.cpu())}")
print(f"Test metrics:  R2 = {r2_score(pred_test, y_test.cpu())}")

Train metrics: R2 = 0.521617432421482
Test metrics:  R2 = 0.23031468151965195


In [7]:
model = Net(input_size=X.shape[1], device=device)
loss_fn = nn.MSELoss()

opt = torch_numopt.ConjugateGradientLS(
    model=model,
    cg_method="PR"
)

model, loss_history = train_loop(
    model,
    loss_fn,
    opt,
    data_loader,
    epochs=1000,
    max_patience=50
)

epoch:  0, loss: 0.6158315539360046


epoch:  1, loss: 0.38534727692604065
epoch:  2, loss: 0.21919038891792297
epoch:  3, loss: 0.12446403503417969
epoch:  4, loss: 0.07669460028409958
epoch:  5, loss: 0.054847076535224915
epoch:  6, loss: 0.04575173556804657
epoch:  7, loss: 0.042307257652282715
epoch:  8, loss: 0.04112011939287186
epoch:  9, loss: 0.040746163576841354
epoch:  10, loss: 0.04063670337200165
epoch:  11, loss: 0.040605057030916214
epoch:  12, loss: 0.04059423506259918
epoch:  13, loss: 0.04057310149073601
epoch:  14, loss: 0.040563296526670456
epoch:  15, loss: 0.04055997356772423
epoch:  16, loss: 0.04052472114562988
epoch:  17, loss: 0.040501464158296585
epoch:  18, loss: 0.040497031062841415
epoch:  19, loss: 0.04046763479709625
epoch:  20, loss: 0.04046233743429184
epoch:  21, loss: 0.040455762296915054
epoch:  22, loss: 0.04043718799948692
epoch:  23, loss: 0.040433429181575775
epoch:  24, loss: 0.0403934083878994
epoch:  25, loss: 0.040356412529945374
epoch:  26, loss: 0.04032755270600319
epoch:  27, 

KeyboardInterrupt: 

In [ ]:
pred_train = model.forward(X_train).cpu().detach()
pred_test = model.forward(X_test).cpu().detach()
print(f"Train metrics: R2 = {r2_score(pred_train, y_train.cpu())}")
print(f"Test metrics:  R2 = {r2_score(pred_test, y_test.cpu())}")

Train metrics: R2 = 0.7392516555973081
Test metrics:  R2 = 0.6229048412077907


In [ ]:
model = Net(input_size=X.shape[1], device=device)
loss_fn = nn.MSELoss()

opt = torch_numopt.ConjugateGradientLS(
    model=model,
    cg_method="PRP+"
)

model, loss_history = train_loop(
    model,
    loss_fn,
    opt,
    data_loader,
    epochs=1000,
    max_patience=50
)

epoch:  0, loss: 0.37063997983932495
epoch:  1, loss: 0.22844940423965454
epoch:  2, loss: 0.1380610316991806
epoch:  3, loss: 0.08727090805768967
epoch:  4, loss: 0.06057443469762802
epoch:  5, loss: 0.04697953164577484
epoch:  6, loss: 0.04024083912372589
epoch:  7, loss: 0.03697335347533226
epoch:  8, loss: 0.03541460633277893
epoch:  9, loss: 0.03467869013547897
epoch:  10, loss: 0.03433258458971977
epoch:  11, loss: 0.034169137477874756
epoch:  12, loss: 0.03409035876393318
epoch:  13, loss: 0.034050602465867996
epoch:  14, loss: 0.03402874246239662
epoch:  15, loss: 0.034015581011772156
epoch:  16, loss: 0.033971380442380905
epoch:  17, loss: 0.03391291946172714
epoch:  18, loss: 0.033894941210746765
epoch:  19, loss: 0.033880095928907394
epoch:  20, loss: 0.033834632486104965
epoch:  21, loss: 0.033828284591436386
epoch:  22, loss: 0.033782392740249634
epoch:  23, loss: 0.03377601504325867
epoch:  24, loss: 0.033727604895830154
epoch:  25, loss: 0.033721230924129486
epoch:  26, 

In [ ]:
pred_train = model.forward(X_train).cpu().detach()
pred_test = model.forward(X_test).cpu().detach()
print(f"Train metrics: R2 = {r2_score(pred_train, y_train.cpu())}")
print(f"Test metrics:  R2 = {r2_score(pred_test, y_test.cpu())}")

Train metrics: R2 = 0.7181293619807423
Test metrics:  R2 = 0.5806095658570554


In [ ]:
model = Net(input_size=X.shape[1], device=device)
loss_fn = nn.MSELoss()

opt = torch_numopt.ConjugateGradientLS(
    model=model,
    cg_method="HS"
)

model, loss_history = train_loop(
    model,
    loss_fn,
    opt,
    data_loader,
    epochs=1000,
    max_patience=50
)

epoch:  0, loss: 0.41564005613327026
epoch:  1, loss: 0.2258436232805252
epoch:  2, loss: 0.07168033719062805
epoch:  3, loss: 0.03826095908880234
epoch:  4, loss: 0.03415548428893089
epoch:  5, loss: 0.03374819457530975
epoch:  6, loss: 0.03369258716702461
epoch:  7, loss: 0.033613353967666626
epoch:  8, loss: 0.03354550898075104
epoch:  9, loss: 0.033526595681905746
epoch:  10, loss: 0.0334368459880352
epoch:  11, loss: 0.03342472016811371
epoch:  12, loss: 0.03341350704431534
epoch:  13, loss: 0.03328755497932434
epoch:  14, loss: 0.03328755497932434
epoch:  15, loss: 0.033215027302503586
epoch:  16, loss: 0.0331740602850914
epoch:  17, loss: 0.033159371465444565
epoch:  18, loss: 0.033015768975019455
epoch:  19, loss: 0.033015768975019455
epoch:  20, loss: 0.03293438255786896
epoch:  21, loss: 0.03288647159934044
epoch:  22, loss: 0.03286967799067497
epoch:  23, loss: 0.03270181640982628
epoch:  24, loss: 0.03270181640982628
epoch:  25, loss: 0.032589688897132874
epoch:  26, loss: 

KeyboardInterrupt: 

In [ ]:
pred_train = model.forward(X_train).cpu().detach()
pred_test = model.forward(X_test).cpu().detach()
print(f"Train metrics: R2 = {r2_score(pred_train, y_train.cpu())}")
print(f"Test metrics:  R2 = {r2_score(pred_test, y_test.cpu())}")

Train metrics: R2 = 0.7636985778808594
Test metrics:  R2 = 0.6484634280204773


In [ ]:
model = Net(input_size=X.shape[1], device=device)
loss_fn = nn.MSELoss()

opt = torch_numopt.ConjugateGradientLS(
    model=model,
    cg_method="DY"
)

model, loss_history = train_loop(
    model,
    loss_fn,
    opt,
    data_loader,
    epochs=1000,
    max_patience=50
)

epoch:  0, loss: 0.1565745323896408
epoch:  1, loss: 0.0898767039179802
epoch:  2, loss: 0.0898767039179802
epoch:  3, loss: 0.037868600338697433
epoch:  4, loss: 0.037722550332546234
epoch:  5, loss: 0.037722546607255936
epoch:  6, loss: 0.03768780454993248
epoch:  7, loss: 0.037550222128629684
epoch:  8, loss: 0.037550222128629684
epoch:  9, loss: 0.03742041438817978
epoch:  10, loss: 0.03731083869934082
epoch:  11, loss: 0.03731083869934082
epoch:  12, loss: 0.03722041845321655
epoch:  13, loss: 0.037189047783613205
epoch:  14, loss: 0.037189047783613205
epoch:  15, loss: 0.037156298756599426
epoch:  16, loss: 0.03709058091044426
epoch:  17, loss: 0.03709058091044426
epoch:  18, loss: 0.03706969693303108
epoch:  19, loss: 0.037012938410043716
epoch:  20, loss: 0.037012938410043716
epoch:  21, loss: 0.03698434680700302
epoch:  22, loss: 0.036922916769981384
epoch:  23, loss: 0.036922916769981384
epoch:  24, loss: 0.036891840398311615
epoch:  25, loss: 0.03681834414601326
epoch:  26, 

In [ ]:
pred_train = model.forward(X_train).cpu().detach()
pred_test = model.forward(X_test).cpu().detach()
print(f"Train metrics: R2 = {r2_score(pred_train, y_train.cpu())}")
print(f"Test metrics:  R2 = {r2_score(pred_test, y_test.cpu())}")

Train metrics: R2 = 0.8116456866264343
Test metrics:  R2 = 0.7160452604293823
